# Analisi degli esperimenti
Analisi degli artefatti aggregati prodotti sul cluster.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from tesi_qml.evaluation.statistics import aggregate_runs, paired_model_comparison

PROJECT_ROOT = Path(os.environ.get("TESI_PROJECT_ROOT", Path.cwd())).resolve()
if not (PROJECT_ROOT / "configs").is_dir() and (PROJECT_ROOT.parent / "configs").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
runs = pd.read_parquet(PROJECT_ROOT / "results/aggregate/runs.parquet")
runs.head()

## Risultati aggregati

In [ ]:
summary = aggregate_runs(runs, metrics=("accuracy", "balanced_accuracy", "f1_macro"))
display(summary)

## Confronti appaiati
Sostituire i nomi dei modelli solo dopo aver congelato il protocollo di confronto.

In [ ]:
comparison_runs = runs.loc[
    (runs["n_qubits"] == 4) & (runs["n_layers"] == 1) & (runs["subset_size"].astype(str) == "full")
]
available = set(comparison_runs["model"])
if {"PL", "B2"} <= available:
    comparison = paired_model_comparison(comparison_runs, "PL", "B2", "f1_macro", seed=42)
    display(pd.Series(comparison))
else:
    print("Eseguire prima entrambi i modelli appaiati PL e B2.")

## Analisi 4 vs 8 qubit

In [ ]:
display(runs.groupby(["model", "n_qubits"], dropna=False)["f1_macro"].agg(["mean", "std", "count"]))